In [ ]:
from finbourne_sdk_utils.jupyter_tools import toggle_code

"""Creating holdings in LUSID

Demonstrates how to load transactions based on custom transaction types and then compute the subsequent holdings.

Attributes
----------
properties
transaction configuration
transactions
"""

toggle_code("Toggle Docstring")

## How do I create holdings in LUSID?

This notebook provides a reference implementation to the ["How do I create holdings in LUSID?"](https://support.finbourne.com/how-do-i-create-holdings) document which demonstrates the process of loading transactions into LUSID to produce holdings.

While we use Python to interact with the LUSID API here, the models and methods are also available through our other maintained SDKs in [C#](https://github.com/finbourne/lusid-sdk-csharp), [Java](https://github.com/finbourne/lusid-sdk-java) and [JavaScript](https://github.com/finbourne/lusid-sdk-js). 

## Setup LUSID

In [ ]:
import os
import finbourne.sdk.services.lusid as lu
import json
import finbourne.sdk.services.lusid.models as models
import pandas as pd
from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
from finbourne.sdk.exceptions import ApiException
from pathlib import Path
import pickle
import pprint
from finbourne_sdk_utils.cocoon.utilities import create_scope_id


# Authenticate our user and create our API client
secrets_path = os.getenv("FBN_SECRETS_PATH")

# Initiate an API Factory which is the client side object for interacting with LUSID APIs
api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook",
)

print("LUSID Environment Initialised")
print(
    "LUSID SDK Version: ",
    api_factory.build(lu.ApplicationMetadataApi)
    .get_lusid_versions()
    .build_version,
)

In [ ]:
# Build the APIs we are going to use

transaction_portfolios_api = api_factory.build(lu.TransactionPortfoliosApi)
portfolios_api = api_factory.build(lu.PortfoliosApi)
property_definitions_api = api_factory.build(lu.PropertyDefinitionsApi)
instruments_api = api_factory.build(lu.InstrumentsApi)
transaction_configuration_api = api_factory.build(lu.TransactionConfigurationApi)
search_api = api_factory.build(lu.SearchApi)

## Load Data files

In [ ]:
txns = pd.read_csv("data/tutorial/tutorial-transactions.csv")
txns

In [ ]:
def create_property_definition(domain, scope, code, data_type):
    try:
        response = property_definitions_api.create_property_definition(
            create_property_definition_request=models.CreatePropertyDefinitionRequest(
                domain=domain,
                scope=scope,
                code=code,
                display_name=code,
                life_time="Perpetual",
                value_required=False,
                data_type_id=models.ResourceId(scope="system", code=data_type)
            )
        )
        return response.key
    except ApiException as e:
        detail = json.loads(e.body)
        if detail["code"] != 124:  # 'PropertyAlreadyExists'
            raise e
        else:
            key = f"{domain}/{scope}/{code}"
            print(f"property {key} already exists")
            return key

## Prepare portfolio and properties

We begin by specifying a scope for loading the data. We also define the portfolio code which will be used to store transactions and holdings. 

In [ ]:
scope = "life_of_a_txn-" + create_scope_id()
portfolio_code = "UK_EQUITY"

print(scope)
print(portfolio_code)

In [ ]:
# Call LUSID to create a property
property = create_property_definition("Portfolio", scope, "portfolio_manager_name", "string")

print(f"Created property {property}")

## Create Portfolio in LUSID

(See [Section 4](https://support.finbourne.com/how-do-i-create-holdings#load-portfolios) of linked tutorial)

In [ ]:
# Create portfolio with properties
subholding_key = {}
created_date = "2010-01-01T00:00:00.000000+00:00"

# create request body
portfolio_request = models.CreateTransactionPortfolioRequest(
    display_name="Life of a transaction",
    code=portfolio_code,
    base_currency="GBP",
    created=created_date,
    properties={
        f"Portfolio/{scope}/portfolio_manager_name": models.ModelProperty(
            key=f"Portfolio/{scope}/portfolio_manager_name",
            value=models.PropertyValue(label_value="Active"),
        )
    },
)

# Upload new portfolio to LUSID
response = transaction_portfolios_api.create_portfolio(
    scope=scope, create_transaction_portfolio_request=portfolio_request
)

created = response.version.effective_from
print(
    f"portfolio '{response.id.code}', in scope {scope} created effective from: "
    f"{created.year}/"
    f"{created.month}/"
    f"{created.day}"
)

## Load Instruments

(See [Section 5](https://support.finbourne.com/how-do-i-create-holdings#load-instruments) of linked tutorial)

In [ ]:
# Call LUSID to create a property
property = create_property_definition("Instrument", scope, "portfolio_manager_name", "string")

print(f"Created property {property}")

Each instrument definition needs to be given an identifier. For equities this could be an Isin or Figi, alternatively for cash this must be "Currency".

In [ ]:
# remove duplicate values
txns.drop_duplicates(subset="client_internal")

# remove cash rows
instruments = txns[txns["client_internal"] != "cash"]
batch_upsert_request = {
    instr["instrument_desc"]: models.InstrumentDefinition(
        name=instr["instrument_desc"],
        identifiers={ "ClientInternal": models.InstrumentIdValue(value=instr["client_internal"]) },
        properties=[
            models.ModelProperty(
                key=f"Instrument/{scope}/portfolio_manager_name",
                value=models.PropertyValue(label_value=instr["portfolio_manager_name"]),
            )
        ],
    )
    for row, instr in instruments.iterrows()
}

# Upsert new instruments to LUSID
instrument_response = instruments_api.upsert_instruments(
    request_body=batch_upsert_request
)

# check response was successful

if len(instrument_response.failed) > 0:
    raise AssertionError("Instruments upsert failed. Inspect response for more detail")

## Loading a Transaction File

(See [Section 6](https://support.finbourne.com/how-do-i-create-holdings#load-txn-file) of linked tutorial)

In [ ]:
# Call LUSID to create a property
property = create_property_definition("Transaction", scope, "portfolio_manager_name", "string")

print(f"Created property {property}")

In [ ]:
# Upsert transactions
transactions_request = []
txn_response = []
for row, txn in txns.iterrows():

    if txn["client_internal"] == "cash":
        instrument_identifier = {"Instrument/default/Currency": "GBP"}
    else:
        instrument_identifier = {
            "Instrument/default/ClientInternal": txn["client_internal"]
        }

    # build request body
    transactions_request.append(
        models.TransactionRequest(
            transaction_id=txn["txn_id"],
            type=txn["transaction_type"],
            instrument_identifiers=instrument_identifier,
            transaction_date=txn["trade_date"],
            settlement_date=txn["trade_date"],
            units=txn["quantity"],
            transaction_price=models.TransactionPrice(price=txn["price"], type="Price"),
            total_consideration=models.CurrencyAndAmount(
                amount=txn["net_money"], currency=txn["instrument_currency"]
            ),
            properties={
                f"Transaction/{scope}/portfolio_manager_name": models.PerpetualProperty(
                    key=f"Transaction/{scope}/portfolio_manager_name",
                    value=models.PropertyValue(
                        label_value=txn["portfolio_manager_name"]
                    ),
                )
            },
        )
    )

    # Make Upsert Transactions call to LUSID
    txn_response.append(
        transaction_portfolios_api.upsert_transactions(
            scope=scope, code=portfolio_code, transaction_request=transactions_request
        )
    )

print(f"{len(txn_response)} transactions upserted")

## Configuring Transaction Types

(See [Section 7](https://support.finbourne.com/how-do-i-create-holdings#config-txn-types) of linked tutorial)

In this section we configure some new transaction types. LUSID already comes pre-configured with some commonly used transaction types (Buy, Sell etc) however we configure the new transaction types of <b>Tutorial-Buy</b> and <b>Tutorial-B</b> below to demonstrate the configurability of the movements engine. 

####  <font color='darkred'>WARNING: Amending Transaction Types can have an unintended system wide impact</font> 

Create the property used in mapping for the custom side

In [ ]:
create_property_definition("Transaction", scope, "Comms2", "number")

In [ ]:
# Construct the default side definitions that, unless removed, can be found in the default scope.
default_side_definitions = [
    models.SidesDefinitionRequest(
        side="Side1", 
        side_request=models.SideDefinitionRequest(
            security="Txn:LusidInstrumentId",
            currency="Txn:TradeCurrency",
            rate="Txn:TradeToPortfolioRate",
            units="Txn:Units",
            amount="Txn:TradeAmount")),
    models.SidesDefinitionRequest(
        side="Side2", 
        side_request=models.SideDefinitionRequest(
            security="Txn:SettleCcy",
            currency="Txn:SettlementCurrency",
            rate="SettledToPortfolioRate",
            units="Txn:TotalConsideration",
            amount="Txn:TotalConsideration"))
]

# Create a list of custom sides which will be used on the new transaction type
side_list = [
        models.SidesDefinitionRequest(
            side="Tutorial-Side1",
            side_request = models.SideDefinitionRequest(
                security="Txn:LusidInstrumentId",
                currency="Txn:TradeCurrency",
                rate="Txn:TradeToPortfolioRate",
                units="Txn:Units",
                amount="Txn:TradeAmount"
            )
        ),
        models.SidesDefinitionRequest(
            side="Tutorial-Side2",
            side_request = models.SideDefinitionRequest(
                security="Txn:SettleCcy",
                currency="Txn:SettlementCurrency",
                rate="SettledToPortfolioRate",
                units="Txn:TotalConsideration",
                amount="Txn:TotalConsideration"
            )
        ),
        models.SidesDefinitionRequest(
            side="Tutorial-TradeCommissions",
            side_request = models.SideDefinitionRequest(
                security="Txn:SettleCcy",
                currency="Txn:SettlementCurrency",
                rate="SettledToPortfolioRate",
                units=f"Transaction/{scope}/Comms2",
                amount=f"Transaction/{scope}/Comms2"
            )
        )
    ]

# Set both the default and non-default sides in the non-default scope
transaction_configuration_api.set_side_definitions(sides_definition_request = default_side_definitions + side_list , scope = scope)

# Add default transaction types
default_transaction_mapping=open('data/default_transaction_mapping.json').read()
default_transaction_mapping = json.loads(default_transaction_mapping)

def map_properties(properties):
    return {property["key"]: models.PerpetualProperty(key=property["key"], value=models.PropertyValue(label_value=property["value"])) for property in properties}
def map_alias(alias):
    return models.TransactionTypeAlias(type=alias["type"], description=alias["description"], transaction_class=alias["transactionClass"], transaction_roles=alias["transactionRoles"])
def map_movement(movement):
    return models.TransactionTypeMovement(movement_types=movement["movementTypes"], side=movement["side"], direction=movement["direction"], properties=map_properties(movement["properties"]))
def map_transaction_type_request(transaction_type_request):
    return models.TransactionTypeRequest(
        aliases=[map_alias(alias) for alias in transaction_type_request["aliases"]],
        movements=[map_movement(movement) for movement in transaction_type_request["movements"]],
        properties=map_properties(transaction_type_request["properties"]))

for configuration in default_transaction_mapping:
    transaction_type_requests = [map_transaction_type_request(transaction_type_request) for transaction_type_request in configuration["transactionTypeRequests"]]
    
    # Call LUSID to set your configuration for our transaction types
    transaction_configuration_api.set_transaction_type_source(
        source=configuration["source"],
        transaction_type_request=transaction_type_requests,
        scope=scope
    )

In [ ]:
# Create the new transaction types using the new sides

try:
    transaction_configuration_api.set_transaction_type(
        source="default",
        type="Tutorial-Buy",
        transaction_type_request=models.TransactionTypeRequest(
            aliases=[
                models.TransactionTypeAlias(
                    type="Tutorial-Buy",
                    description="A purchase transaction from System X",
                    transaction_class="Basic",
                    transaction_roles="LongLonger",
                )
            ],
            movements=[
                models.TransactionTypeMovement(
                    movement_types="StockMovement",
                    side="Tutorial-Side1",
                    direction=1,
                    properties={},
                    mappings=[],
                ),
                models.TransactionTypeMovement(
                    movement_types="CashCommitment",
                    side="Tutorial-Side2",
                    direction=-1,
                    properties={},
                    mappings=[],
                ),
                models.TransactionTypeMovement(
                    movement_types="CashCommitment",
                    side="Tutorial-TradeCommissions",
                    direction=-1,
                    properties={},
                    mappings=[
                        models.TransactionTypePropertyMapping(
                            property_key=f"Transaction/{scope}/Broker_2",
                            set_to="Commission",
                        )
                    ],
                ),
            ],
        ),
        scope = scope
    )
    transaction_configuration_api.set_transaction_type(
        source="alt1",
        type="Tutorial-B",
        transaction_type_request=models.TransactionTypeRequest(
            aliases=[
                models.TransactionTypeAlias(
                    type="Tutorial-B",
                    description="A purchase transaction from System X",
                    transaction_class="Basic",
                    transaction_roles="LongLonger",
                ),
            ],
            movements=[
                models.TransactionTypeMovement(
                    movement_types="StockMovement",
                    side="Tutorial-Side1",
                    direction=1,
                    properties={},
                    mappings=[],
                ),
                models.TransactionTypeMovement(
                    movement_types="CashCommitment",
                    side="Tutorial-Side2",
                    direction=-1,
                    properties={},
                    mappings=[],
                ),
                models.TransactionTypeMovement(
                    movement_types="CashCommitment",
                    side="Tutorial-TradeCommissions",
                    direction=-1,
                    properties={},
                    mappings=[
                        models.TransactionTypePropertyMapping(
                            property_key=f"Transaction/{scope}/Broker_2",
                            set_to="Commission",
                        )
                    ],
                ),
            ],
        ),
        scope = scope
    )

except ApiException as e:
    print(json.loads(e.body)["title"])


# Call LUSID to update the transaction type scope of your portfolio 
patch_document = [
    {
        "value": scope,
        "path": "/transactiontypescope",
        "op": "add"
    }
]
patch_response = api_factory.build(lu.TransactionPortfoliosApi).patch_portfolio_details(
    scope=scope,
    code=portfolio_code,
    operation=patch_document)

## Get Holdings

(See [Section 8](https://support.finbourne.com/how-do-i-create-holdings#get-holdings) of linked tutorial)

In [ ]:
# Prints the name and a quick summary from a get_holdings() response
def display_holdings_summary(response):
    # inspect holdings response for today
    hld = [i for i in response.values]

    names = []
    amount = []
    units = []
    shks = []

    for item in hld:

        names.append(item.properties["Instrument/default/Name"].value.label_value)
        amount.append(item.cost.amount)
        units.append(item.units)
        shks.append(
            [
                item.sub_holding_keys[key].value.label_value
                for key in item.sub_holding_keys.keys()
            ]
        )

    data = {"names": names, "amount": amount, "units": units, "shks": shks}

    summary = pd.DataFrame(data=data)
    return summary

#### Get holdings with todays effective date

In [ ]:
holdings_response_today = transaction_portfolios_api.get_holdings(
    scope=scope, code=portfolio_code, property_keys=["Instrument/default/Name"]
)

summary = display_holdings_summary(holdings_response_today)
summary

#### Get holdings effective 1st January 2020 (the trade date of the first transactions)

In [ ]:
holdings_response_1st_jan = transaction_portfolios_api.get_holdings(
    scope=scope,
    code=portfolio_code,
    effective_at="2020-01-01T00:00:01.0000000+00:00",
    property_keys=["Instrument/default/Name"],
)

display_holdings_summary(holdings_response_1st_jan)

#### Get holdings effective 1st March (after the trade date of the last transaction)

In [ ]:
holdings_response_1st_Mar = transaction_portfolios_api.get_holdings(
    scope=scope,
    code=portfolio_code,
    property_keys=["Instrument/default/Name"],
    effective_at="2020-03-01T00:00:01.0000000+00:00",
)

display_holdings_summary(holdings_response_1st_Mar)

## Sub-holding Keys

(See [Section 9](https://support.finbourne.com/how-do-i-create-holdings#shk) of linked tutorial)

In order to segregate our new portfolio that includes Sub-holding Keys (SHKs), we will use a different scope.

In [ ]:
txns_strat = pd.read_csv("data/tutorial/txn_strategy.csv")

In [ ]:
scope_shk = scope + "-stratategy-tagging"
print(scope_shk)

#### create the property for `StrategyTag`

In [ ]:
# Call LUSID to create our new property, this property willbe used as our SHK

property = create_property_definition("Transaction", scope_shk, "StrategyTag", "string")

print(f"Created property {property}")

#### Assign property to portfolio (or derived portfolio) as SHK

In [ ]:
# Create portfolio with properties
created_date = "2010-01-01T00:00:00.000000+00:00"

# create request body
portfolio_request = models.CreateTransactionPortfolioRequest(
    display_name="Life of a transaction - stratategy tagging",
    code=portfolio_code,
    base_currency="GBP",
    created=created_date,
    sub_holding_keys=[f"Transaction/{scope_shk}/StrategyTag"],
    transaction_type_scope=scope
)

# Upload new portfolio to LUSID
response = transaction_portfolios_api.create_portfolio(
    scope=scope_shk, create_transaction_portfolio_request=portfolio_request
)

created = response.version.effective_from
print(
    f"portfolio '{response.id.code}', in scope {scope_shk} created effective from: "
    f"{created.year}/"
    f"{created.month}/"
    f"{created.day}"
)

#### Upload transactions with properties for SHK

In [ ]:
# Upsert transactions
transactions_request = []
txn_response = []
for row, txn in txns_strat.iterrows():

    if txn["client_internal"] == "cash":
        instrument_identifier = {"Instrument/default/Currency": "GBP"}
    else:
        instrument_identifier = {
            "Instrument/default/ClientInternal": txn["client_internal"]
        }

    # build request body
    transactions_request.append(
        models.TransactionRequest(
            transaction_id=txn["txn_id"],
            type=txn["transaction_type"],
            instrument_identifiers=instrument_identifier,
            transaction_date=txn["trade_date"],
            settlement_date=txn["trade_date"],
            units=txn["quantity"],
            transaction_price=models.TransactionPrice(price=txn["price"], type="Price"),
            total_consideration=models.CurrencyAndAmount(
                amount=txn["net_money"], currency=txn["instrument_currency"]
            ),
            properties={
                f"Transaction/{scope_shk}/StrategyTag": models.PerpetualProperty(
                    key=f"Transaction/{scope_shk}/StrategyTag",
                    value=models.PropertyValue(label_value=txn["StrategyTag"]),
                ),
            },
        )
    )

    # Make Upsert Transactions call to LUSID
    txn_response.append(
        transaction_portfolios_api.upsert_transactions(
            scope=scope_shk,
            code=portfolio_code,
            transaction_request=transactions_request,
        )
    )

print(f"{len(txn_response)} transactions upserted")

### Get Holdings from new portfolio with SHK

Now when we get holdings, the result will be segregated by any specified Sub-holding Keys that we specified on the portfolio. 

In [ ]:
holdings_response_1st_Mar = transaction_portfolios_api.get_holdings(
    scope=scope_shk,
    code=portfolio_code,
    property_keys=["Instrument/default/Name",],
    effective_at="2020-03-01T00:00:01.0000000+00:00",
)

display_holdings_summary(holdings_response_1st_Mar)

In [ ]:
holdings_response_1st_jan = transaction_portfolios_api.get_holdings(
    scope=scope_shk,
    code=portfolio_code,
    effective_at="2020-01-01T00:00:01.0000000+00:00",
    property_keys=["Instrument/default/Name",],
)

display_holdings_summary(holdings_response_1st_jan)

## Footnote: Resetting the original transaction configuration

Transaction types are system wide settings. Therefore you might want to delete or reset the ones created above. If so, you can create a new [CreateConfigurationTransactionType](https://www.lusid.com/docs/api/#operation/CreateConfigurationTransactionType) request without the new types created above. One way of doing this is with the [lusidtools CLI](https://github.com/finbourne/lusid-python-tools/wiki/lusidtools-CLI).


## Tear down

Remove example data

In [ ]:
try:
    property_defns = search_api.search_properties(search=f"\"{scope}\"")

    for defn in property_defns.values:
        property_definitions_api.delete_property_definition(defn.domain, defn.scope, defn.code)
        display(f"deleted property {defn.key}")
except Exception as e:
    display(e)

In [ ]:
try:
    portfolios_api.delete_portfolio(scope, portfolio_code)
    display(f"deleted portfolio {scope}/{portfolio_code}")
    
    portfolios_api.delete_portfolio(scope_shk, portfolio_code)
    display(f"deleted portfolio {scope}/{scope_shk}")
    
except ApiException as e:
    detail = json.loads(e.body)
    if detail["code"] != 109:  # 'PortfolioNotFound'
        raise e